# HuBMAP Vasculature — YOLO26-seg 兩階段 4-fold 推理 (Kaggle)

Stage1(ds2 粗練)→ Stage2(ds1 精練)的 YOLO26x-seg,推理用 **4-fold WBF + 4-way TTA**
(重用 repo 既有 `predict_ensemble.py` + `train.py submit`)。

離線 wheels、Internet OFF 可提交。把 `DATASET_NAME` 改成你上傳的 weights dataset slug 後段名稱。
之後可把此 `submission.csv`(或 `preds/*.npz`)與第一名 InternImage 在 Kaggle 端再 ensemble。

In [ ]:
# ===== Cell 1: 路徑設定 =====
import os, shutil, subprocess, sys, time
from pathlib import Path

DATASET_NAME = "hubmap-yolo26-2stage"
COMP_NAME = "hubmap-hacking-the-human-vasculature"

DATASET_DIR = Path(f"/kaggle/input/{DATASET_NAME}")
COMP_DIR = Path(f"/kaggle/input/{COMP_NAME}")
WORK = Path("/kaggle/working")

assert DATASET_DIR.exists(), f"找不到 {DATASET_DIR}，請確認 dataset 已 attach"
assert COMP_DIR.exists(), f"找不到比賽資料 {COMP_DIR}"

print("Dataset 內容：")
for p in sorted(DATASET_DIR.iterdir()):
    print(" ", p.name)

In [ ]:
# ===== Cell 2: 複製推理程式到 working =====
# predict_ensemble.py 會 import train / dataprocess，三個都要在同一層
for name in ("train.py", "dataprocess.py", "predict_ensemble.py"):
    shutil.copy(DATASET_DIR / "code" / name, WORK / name)
    print("  ", WORK / name)

In [ ]:
# ===== Cell 3: 離線安裝(Internet OFF 友善)=====
WHEELS = DATASET_DIR / "wheels"
if WHEELS.exists() and any(WHEELS.glob("*.whl")):
    !pip install --no-index --find-links={WHEELS} ultralytics opencv-python-headless pycocotools --no-deps -q
else:
    print("(無 wheels/，假設環境已含 ultralytics)")

import importlib
for pkg in ("ultralytics", "cv2", "pycocotools"):
    try:
        m = importlib.import_module(pkg)
        print(f"  ✓ {pkg} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  ✗ {pkg}: {e}")

In [ ]:
# ===== Cell 4: 收集 fold 權重(支援 1~4 個)=====
yolo_weights = sorted((DATASET_DIR / "yolo").glob("fold*.pt"))
assert yolo_weights, f"找不到 fold 權重於 {DATASET_DIR / 'yolo'}"
print("YOLO fold 權重：", [p.name for p in yolo_weights])

In [ ]:
# ===== Cell 5: 4-fold WBF + 4-way TTA 推理 → preds/*.npz =====
os.chdir(WORK)
out_dir = WORK / "preds"
out_dir.mkdir(exist_ok=True)

cmd = [
    sys.executable, "predict_ensemble.py",
    "--weights", *[str(p) for p in yolo_weights],
    "--src", str(COMP_DIR / "test"),
    "--meta", str(COMP_DIR / "tile_meta.csv"),
    "--out", str(out_dir),
    "--conf", "0.05", "--iou", "0.7", "--wbf-iou", "0.55",
    "--device", "0",
]
print(" ".join(cmd))
t0 = time.time()
subprocess.run(cmd, check=True, cwd=str(WORK))
print(f"\n推理用時 {time.time() - t0:.1f}s")

In [ ]:
# ===== Cell 6: 產 submission.csv(RLE) =====
subprocess.run([
    sys.executable, "train.py", "submit",
    "--preds", str(out_dir),
    "--out", str(WORK / "submission.csv"),
], check=True, cwd=str(WORK))

In [ ]:
# ===== Cell 7: 檢查 submission =====
import pandas as pd
df = pd.read_csv(WORK / "submission.csv")
print(df.shape)
print("非空 prediction：", (df["prediction_string"].fillna("") != "").sum())
df.head(3)